# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant/) library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure that the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant schema.

The `mlcroissant` API loads Croissant schema objects such as record sets and fields; each is uniquely identified by its `@id`.

In [ ]:
# List all available record sets by @id and name
print("Available record sets (by @id):")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', rs['@id'])}")

# For each record set, list its fields (columns) by @id
for rs in record_sets:
    print(f"\nFields for RecordSet '@id': {rs['@id']}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
            print(f"  - {field_id}")
    else:
        print("  (No fields found)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below we extract all records for the main data table. If there are multiple record sets, they can be loaded individually, each referenced by its unique `@id`.

In [ ]:
# Collect all record sets by @id
record_set_ids = [rs['@id'] for rs in record_sets]

# Build DataFrames for each record set
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# If there is only one record set, show its columns and head
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Columns for RecordSet '@id': {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets were found in the schema.")

## 4. Exploratory Data Analysis (EDA)

Let's apply common data processing steps, such as filtering and normalization, on a numeric field from the main record set.

Please update the `numeric_field_id` and `group_field_id` below according to actual available fields, as listed above.

In [ ]:
# === Set these based on the fields listed above ===
# Example field IDs (edit as needed):
main_record_set = main_record_set_id

# For demonstration, try to guess a numeric field (replace with the appropriate field @id)
numeric_field_id = None
for col in dataframes[main_record_set].columns:
    # Naively look for potential numeric column names
    if any(keyword in col.lower() for keyword in ['age', 'interval', 'number', 'duration']):
        numeric_field_id = col
        break
if numeric_field_id is None:
    # If we didn't find, fall back to asking the user
    print('Please choose a numeric field id from:', list(dataframes[main_record_set].columns))
else:
    print(f"Using numeric field: {numeric_field_id}")

# For group field, try to find by name
group_field_id = None
for col in dataframes[main_record_set].columns:
    if any(keyword in col.lower() for keyword in ['sex','sex','group','category','type']):
        group_field_id = col
        break
if group_field_id is None:
    print('Please choose a group field id from:', list(dataframes[main_record_set].columns))
else:
    print(f"Using group field: {group_field_id}")

# Proceed if we have a numeric field
if numeric_field_id:
    df = dataframes[main_record_set].copy()
    # Ensure numeric conversion
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    threshold = df[numeric_field_id].quantile(0.5)  # E.g., median threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by group_field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df)
else:
    print("No numeric field selected for EDA.")

## 5. Visualization

Visualize the distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution (if selected)
if numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion

In this notebook, we've:
- Loaded the dataset metadata and records using the Croissant schema with `mlcroissant`.
- Explored record sets, fields, and identified them by their `@id`.
- Loaded data into pandas DataFrames, performed simple EDA (filtering, normalization, and group statistics).
- Visualized the distributions of key numeric fields.

This workflow provides a reproducible template for further clinical, statistical, or ML model development on the FAIR² dataset.